# 03 — SGLang as an alternative serving backend

The agent only talks to an OpenAI-compatible URL, so pointing it at SGLang
instead of vLLM needs no code change. This notebook runs the same eval suite
and the same time-to-first-token trace on SGLang, for a side-by-side comparison
with notebooks 01 and 02.

Use a fresh runtime (Runtime > Disconnect and delete runtime) so the GPU is empty.

In [ ]:
# --- Setup: get the code and install the agent-side packages ---
REPO_URL = "https://github.com/sauerpatchkid/Portfolio_Assistant.git"
REPO_DIR = "/content/portfolio-assistant"

import os

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull -q
!pip install -q -r requirements.txt

os.makedirs("results", exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- Serving environment: SGLang in its own virtualenv ---
VENV = "/content/venv-sglang"

!pip install -q uv
if not os.path.exists(VENV):
    !uv venv {VENV} --python 3.12 -q
    !uv pip install -q --python {VENV}/bin/python "sglang[all]"
os.environ["SERVING_VENV"] = VENV

!uv pip freeze --python {VENV}/bin/python | grep -iE "^(sglang|torch|transformers)==" | tee results/versions_sglang.txt

In [ ]:
from serving.launch import start_server, stop_server

MODEL = "Qwen/Qwen3-8B-AWQ"
PROMPT = "v1"

server = start_server("sglang", MODEL, log_name="sglang_large")

In [ ]:
!python -m scripts.smoke_check --name smoke_sglang_large --model {MODEL}

## TTFT on the fixed trace (run first, while the server's cache is still empty)

In [ ]:
!python -m bench.ttft --name ttft_sglang --model {MODEL}

## Eval suite

In [ ]:
!python -m eval.run_eval --name eval_sglang_8b_{PROMPT}_c16 --model {MODEL} --prompt {PROMPT} --concurrency 16
!python -m eval.run_eval --name eval_sglang_8b_{PROMPT}_c1 --model {MODEL} --prompt {PROMPT} --concurrency 1

In [ ]:
stop_server(server)

## Results

In [ ]:
import pandas as pd

display(pd.read_csv("results/summary.csv")[["name", "concurrency", "correct", "accuracy_pct", "errors",
                                            "wall_s", "completion_tokens_per_s"]])
display(pd.read_csv("results/ttft_summary.csv"))

In [ ]:
# --- Download the results ---
!zip -qr /content/results_03_sglang.zip results
from google.colab import files
files.download("/content/results_03_sglang.zip")